<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/HEADSWAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#@title 1. Install dependencies
!pip install -q --upgrade "torchao>=0.16.0" "diffusers>=0.33.0" transformers accelerate peft safetensors pillow opencv-python-headless fastapi uvicorn python-multipart huggingface_hub

from IPython.display import clear_output
clear_output()
print("✅ Installed")

✅ Installed


In [13]:
#@title 2. Download & verify Qwen-Image 2.1 + BFS Head Swap LoRA weights
import gc
import os
import torch
from huggingface_hub import hf_hub_download, snapshot_download

torch.cuda.empty_cache()
gc.collect()

LORA_DIR = "/content/loras"
os.makedirs(LORA_DIR, exist_ok=True)

print("Downloading/verifying base model (Qwen/Qwen-Image-2.1)...")
snapshot_download(
    repo_id="Qwen/Qwen-Image-2.1",
    ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
)

print("Downloading BFS Head Swap LoRA (Alissonerdx/BFS-Best-Face-Swap)...")
lora_path = hf_hub_download(
    repo_id="Alissonerdx/BFS-Best-Face-Swap",
    filename="bfs_head_v1.1_qwen_2.1.safetensors",
    local_dir=LORA_DIR,
)

with open(lora_path, "rb") as f:
    header = f.read(16)
if header.startswith(b"{") or header.startswith(b"<"):
    raise RuntimeError(f"Downloaded file is not a valid safetensors binary: {lora_path}")

lora_size_mb = os.path.getsize(lora_path) / (1024 * 1024)
print(f"✅ Weights cached. LoRA path: {lora_path} ({lora_size_mb:.2f} MB)")

Downloading/verifying base model (Qwen/Qwen-Image-2.1)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

✅ Weights cached. LoRA path: /content/loras/bfs_head_v1.1_qwen_2.1.safetensors (248.05 MB)


In [10]:
#@title 3A. Clear kernel VRAM
import gc
import torch

if "pipe" in globals():
    del pipe

gc.collect()
torch.cuda.empty_cache()
print("✅ Notebook kernel GPU memory cleared.")

✅ Notebook kernel GPU memory cleared.


In [11]:
%%writefile /content/headswap_server.py
import base64
import gc
import glob
import os
import sys
import traceback
from collections import deque
from contextlib import asynccontextmanager
from io import BytesIO
from typing import Optional

import torch
from diffusers import DiffusionPipeline
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from pydantic import BaseModel
from safetensors.torch import load_file

pipe: Optional[DiffusionPipeline] = None
MAX_SIDE = 1024
LOG_BUFFER = deque(maxlen=200)
LORA_DIR = "/content/loras"


def log_msg(msg: str) -> None:
    line = str(msg)
    LOG_BUFFER.append(line)
    print(line, flush=True)


def find_lora_file(lora_dir: str = LORA_DIR) -> str:
    candidates = sorted(glob.glob(os.path.join(lora_dir, "*.safetensors")))
    if not candidates:
        raise FileNotFoundError(f"No .safetensors LoRA found in {lora_dir}")
    return candidates[0]


def map_lora_state_dict(state_dict: dict) -> dict:
    """Normalizes ComfyUI/Civitai LoRA prefixes while preserving Qwen-Image block names."""
    mapped = {}
    for k, v in state_dict.items():
        nk = k
        if nk.startswith("diffusion_model."):
            nk = nk[len("diffusion_model."):]
        if not nk.startswith("transformer.") and "transformer_blocks." in nk:
            nk = "transformer." + nk
        mapped[nk] = v
    return mapped


def load_pipeline() -> DiffusionPipeline:
    log_msg("Loading Qwen-Image 2.1 + Civitai LoRA in bfloat16 on L4...")
    torch.cuda.empty_cache()
    gc.collect()

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    pipeline = DiffusionPipeline.from_pretrained(
        "Qwen/Qwen-Image-2.1",
        dtype=dtype,
    )

    lora_file = find_lora_file(LORA_DIR)
    log_msg(f"Loading LoRA weights from: {lora_file}")

    try:
        pipeline.load_lora_weights(lora_file)
    except Exception as lora_err:
        log_msg(f"Standard load_lora_weights fallback: {lora_err}")
        raw_sd = load_file(lora_file)
        mapped_sd = map_lora_state_dict(raw_sd)
        pipeline.load_lora_weights(mapped_sd)

    try:
        pipeline.to("cuda")
        log_msg("Loaded pipeline directly onto CUDA (bfloat16).")
    except torch.OutOfMemoryError:
        log_msg("CUDA full-load OOM on L4; enabling bfloat16 model CPU offload...")
        torch.cuda.empty_cache()
        gc.collect()
        pipeline.enable_model_cpu_offload()

    if hasattr(pipeline, "vae") and pipeline.vae is not None:
        if hasattr(pipeline.vae, "enable_tiling"):
            pipeline.vae.enable_tiling()
        if hasattr(pipeline.vae, "enable_slicing"):
            pipeline.vae.enable_slicing()

    torch.cuda.empty_cache()
    gc.collect()
    return pipeline


@asynccontextmanager
async def lifespan(app: FastAPI):
    global pipe
    pipe = load_pipeline()
    log_msg("Model + LoRA loaded and ready")
    yield
    if pipe is not None:
        del pipe
        pipe = None
    torch.cuda.empty_cache()
    gc.collect()


app = FastAPI(lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class HeadSwapRequest(BaseModel):
    body: str
    head: str
    prompt: str = ""
    num_inference_steps: int = 20
    guidance_scale: float = 1.0


def dataurl_to_pil(dataurl: str) -> Image.Image:
    if "," in dataurl:
        dataurl = dataurl.split(",", 1)[1]
    image_data = base64.b64decode(dataurl)
    return Image.open(BytesIO(image_data)).convert("RGB")


def pil_to_dataurl(img: Image.Image) -> str:
    buffered = BytesIO()
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{img_str}"


def resize_max_side(img: Image.Image, max_side: int = MAX_SIDE) -> Image.Image:
    w, h = img.size
    largest = max(w, h)
    if largest <= max_side:
        new_w = max(16, (w // 16) * 16)
        new_h = max(16, (h // 16) * 16)
        if (new_w, new_h) != (w, h):
            return img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        return img
    scale = max_side / float(largest)
    new_w = max(16, (int(round(w * scale)) // 16) * 16)
    new_h = max(16, (int(round(h * scale)) // 16) * 16)
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


@app.get("/health")
def health():
    return {"status": "ok", "loaded": pipe is not None}


@app.get("/logs")
def get_logs():
    return {"logs": list(LOG_BUFFER), "lines": list(LOG_BUFFER)}


@app.post("/headswap")
def headswap(req: HeadSwapRequest):
    try:
        if pipe is None:
            return {"error": "Pipeline not initialized"}

        raw_body = dataurl_to_pil(req.body)
        raw_head = dataurl_to_pil(req.head)

        body_img = resize_max_side(raw_body, MAX_SIDE)
        head_img = resize_max_side(raw_head, MAX_SIDE)

        prompt = req.prompt.strip() or (
            "head_swap: start with the first image as the base image, keeping its lighting, "
            "environment, and background. remove the head from the first image completely and "
            "replace it with the head from the second image, strictly preserving the hair, eye "
            "color, nose structure from the second image. copy the direction of the eye, head "
            "rotation, micro expressions from the first image, high quality, sharp details, 4k"
        )

        steps = max(6, min(30, int(req.num_inference_steps)))
        log_msg(
            f"Running head swap... raw_body={raw_body.size} -> body={body_img.size}, "
            f"raw_head={raw_head.size} -> head={head_img.size}, steps={steps}"
        )

        torch.cuda.empty_cache()
        gc.collect()

        def on_step_end(_pipe, step_index, _timestep, callback_kwargs):
            log_msg(f"Step {step_index + 1}/{steps}")
            return callback_kwargs

        call_kwargs = {
            "prompt": prompt,
            "image": [body_img, head_img],
            "num_inference_steps": steps,
            "height": body_img.height,
            "width": body_img.width,
        }
        if "guidance_scale" in pipe.__call__.__code__.co_varnames:
            call_kwargs["guidance_scale"] = req.guidance_scale
        if "callback_on_step_end" in pipe.__call__.__code__.co_varnames:
            call_kwargs["callback_on_step_end"] = on_step_end

        with torch.inference_mode():
            result = pipe(**call_kwargs).images[0]

        if result.size != raw_body.size:
            result = result.resize(raw_body.size, Image.Resampling.LANCZOS)

        log_msg("Done.")
        return {"image": pil_to_dataurl(result)}

    except Exception as e:
        error_msg = traceback.format_exc()
        log_msg(error_msg)
        return {"error": str(e)}
    finally:
        torch.cuda.empty_cache()
        gc.collect()


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000)

Writing /content/headswap_server.py


In [14]:
#@title 4. Start server + Cloudflare tunnel
import os
import re
import subprocess
import sys
import time
import urllib.request

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart"],
    check=True,
)

SERVER_SCRIPT = "/content/headswap_server.py"
SERVER_LOG = "/content/headswap_server.log"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"

subprocess.run(["pkill", "-f", "headswap_server.py"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            CLOUDFLARED,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

print("Starting Head Swap server...")
with open(SERVER_LOG, "w") as log:
    server_proc = subprocess.Popen(
        [sys.executable, SERVER_SCRIPT],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Waiting for model to load and Uvicorn to bind port 8000...")
model_ready = False
for _ in range(180):
    time.sleep(2)
    if server_proc.poll() is not None:
        print("❌ Server process exited early. Check log below:")
        break
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG) as f:
            content = f.read()
        if "Application startup complete" in content:
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3) as resp:
                    if resp.status == 200 and b'"loaded":true' in resp.read().replace(b" ", b""):
                        print("✅ Model loaded and /health is ready")
                        model_ready = True
                        break
            except Exception:
                pass

if not model_ready:
    print("⚠️ Model did not finish loading in time. Last 30 log lines:")
    with open(SERVER_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("HeadSwap server failed to become ready.")

print("Starting Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

TUNNEL_URL = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        TUNNEL_URL = m.group(0)
        break

if TUNNEL_URL:
    print("\n" + "=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("=" * 60)
    print("\n→ Copy the URL above and paste it into the Head Swap tab on your website.")
else:
    print("❌ Could not get Cloudflare URL. Try re-running this cell.")

Starting Head Swap server...
Waiting for model to load and Uvicorn to bind port 8000...
✅ Model loaded and /health is ready
Starting Cloudflare tunnel...

READY: https://enhancing-lamb-diagram-big.trycloudflare.com

→ Copy the URL above and paste it into the Head Swap tab on your website.


In [21]:
!tail -50 /content/headswap_server.log

INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
Running head swap... raw_body=(853, 1280) -> body=(672, 1024), raw_head=(638, 720) -> head=(624, 720), steps=20
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
  0%|          | 0/20 [00:00<?, ?it/s]INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /logs HTTP/1.1" 200 OK
INFO:     114.125.149.160:0 - "GET /lo